In [60]:
import chipwhisperer as cw, time

scope = cw.scope()
scope.default_setup()
target = cw.target(scope, cw.targets.SimpleSerial)

scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"
target.ser.cwlite_usart.init(baud=115200, stopbits=1, parity="even")
#target.ser.cwlite_usart.init(baud=9600, stopbits=1, parity="even")

# Enter the bootloader: BOOT0 high, then reset
scope.io.pdic = "high"        # drives BOOT0 (use whichever GPIO you wired)
scope.io.nrst = "low"; time.sleep(0.1)
scope.io.nrst = "high_z"; time.sleep(0.1)

# Auto-baud handshake
target.ser.flush()
target.ser.write(b"\x7f")
#print(target.ser.read(1, timeout=1000))   # expect b'\x79'
time.sleep(0.2)
#print(target.in_waiting())          # number of bytes received
#print(target.ser.cwlite_usart.read(1, timeout=1000))
#data = target.read(1, timeout=1000)
#print(repr(data), type(data))
n = target.in_waiting() #if callable(target.in_waiting) else target.in_waiting
print(n)
data = target.read(n, timeout=500) if n else b""
print(data.encode("latin-1").hex())

# Get ID command: 0x02, then its complement 0xFD
#target.ser.write(bytes([0x02, 0xFD]))
#print(target.ser.read(5, timeout=500))   # ACK, N, PID(2 bytes), ACK

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

In [85]:
# not with target but serial
import serial, time

# pins must stay in serial mode so the CDC port is routed to them
scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"

ser = serial.Serial("/dev/ttyACM0", 115200, parity=serial.PARITY_EVEN,                    stopbits=1, bytesize=8, timeout=1)

# enter the bootloader: BOOT0 high, then reset
scope.io.pdic = "high"
scope.io.nrst = "low";  time.sleep(0.1)
scope.io.nrst = "high_z"; time.sleep(0.3)

ser.reset_input_buffer()          # drop any glitch bytes from the reset
ser.write(b"\x7f")
print("handshake:", ser.read(1).hex())          # expect 79

# Get ID: 0x02 + complement 0xFD
ser.write(bytes([0x02, 0xFD]))
print("get id:", ser.read(5).hex())             # expect 79 01 0435 79

handshake: 79
get id: 7901046279


In [62]:
# loopback test
u = target.ser.cwlite_usart
for p in ("even", "none", "odd"):
    u.init(baud=115200, stopbits=1, parity=p)
    u.flush()
    u.write(b"\x79")
    time.sleep(0.1)
    print(p, bytes(u.read(1, timeout=500)).hex())

even fc
none fc
odd fc


In [63]:
# loopback with different baud rates
u = target.ser.cwlite_usart
for b in (9600, 19200, 38400, 57600, 115200, 230400):
    u.init(baud=b, stopbits=1, parity="none")
    u.flush()
    u.write(b"\x79")
    time.sleep(0.1)
    print(b, bytes(u.read(1, timeout=500)).hex())

9600 e0
19200 f0
38400 c0
57600 00
115200 00
230400 00


In [66]:
target.baud = 115200
target.flush(); target.write(b"\x79"); time.sleep(0.1)
print(target.read(1, timeout=500).encode("latin-1").hex())

00


In [72]:
# test connection
scope.io.tio1 = "gpio_low"
scope.io.tio2 = "high_z"
print(scope.io.tio_states)      # note the state of TIO1 and TIO2
scope.io.tio1 = "gpio_high"
print(scope.io.tio_states)

(0, 0, 1, 0)
(1, 1, 1, 0)


In [73]:
scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"
u = target.ser.cwlite_usart
print(type(u))
print(u.init.__doc__)           # shows the accepted argument forms
u.init(baud=115200)
u.flush(); u.write(b"\x79"); time.sleep(0.1)
print(bytes(u.read(1, timeout=500)).hex())

<class 'chipwhisperer.hardware.naeusb.serial.USART'>

        Open the serial port, set baud rate, parity, etc.
        
00


In [76]:
import serial
ports = scope.get_serial_ports()
print(ports)
ser = serial.Serial(ports[0]["port"], 115200, parity=serial.PARITY_EVEN, stopbits=1, bytesize=8, timeout=1)
ser.write(b"\x7f")
print(ser.read(1).hex())

[{'port': '/dev/ttyACM0', 'interface': 1}]
5e


In [77]:
scope.io.tio1 = "gpio_low"
scope.io.tio2 = "high_z"
print(scope.io.tio_states)
scope.io.tio1 = "gpio_high"
print(scope.io.tio_states)
scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"

(0, 0, 1, 0)
(1, 1, 1, 0)


In [82]:
u = target.ser.cwlite_usart
for sb in (1, 2, "1", "2"):
    try:
        u.init(baud=115200, stopbits=sb, parity="none")
        u.flush(); u.write(b"\x79"); time.sleep(0.1)
        print(repr(sb), bytes(u.read(1, timeout=500)).hex())
    except Exception as e:
        print(repr(sb), "error:", e)
        

1 00
2 00
'1' error: Invalid stop-bit 1, must be one of [1, 1.5, 2]
'2' error: Invalid stop-bit 2, must be one of [1, 1.5, 2]


In [79]:
print(cw.__version__)
print(scope.fw_version_str)
help(target.ser.cwlite_usart.init)

7.0.0
1.2.0
Help on method init in module chipwhisperer.hardware.naeusb.serial:

init(baud=115200, stopbits=1, parity='none') method of chipwhisperer.hardware.naeusb.serial.USART instance
    Open the serial port, set baud rate, parity, etc.



In [84]:
ser = serial.Serial("/dev/ttyACM0", 115200, parity=serial.PARITY_EVEN, timeout=1)
ser.write(b"\x79"); print(ser.read(1).hex())

79


In [83]:
for payload in (b"ABC", "ABC", [0x41, 0x42, 0x43]):
    u.flush()
    try:
        u.write(payload)
        time.sleep(0.2)
        print(type(payload).__name__, bytes(u.read(3, timeout=500)).hex())
    except Exception as e:
        print(type(payload).__name__, "error:", e)

bytes 000000
str 000000
list 000000
